# Multi-Image Confidence Coverage Analysis

Sweeps a symmetric confidence margin `T` around 0.5 on the transformer fusion model's pooled per-patient predictions (accept when `p < 0.5-T` or `p > 0.5+T`). 

In [ ]:
import sys, os, glob

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

from model import UnifiedBackboneMulti
from multi_image_dataset import MultiImageFundusDataset
from diagnosis_train_eval import validate_multi

## Configuration

In [ ]:
RUN_ID     = 1
DATASET    = "BRSET"           # "BRSET" or "mBRSET"
MODEL_NAME = "retfound_green"

# Must match the prefix used when the fusion model was trained
CHECKPOINT_PREFIX = f"run{RUN_ID}_{DATASET}_multi_"
N_CHECKPOINTS      = 5

# img_indices_list: list of image-index selections to average over.
#   None  -> all images for the patient (no masking)
#   [0,2] -> 1st and 3rd image; remaining slots zero-masked
# e.g. mBRSET 2-image averaging: [[0, 2], [1, 3]]
IMG_INDICES_LIST = [None]

# T is swept from 0 to 0.5; patient accepted when p < 0.5-T or p > 0.5+T
T_STEP     = 0.01
N_BINS     = 20
PROB_SHIFT = 0.1   # subtracted from raw probabilities to correct for observed overconfidence

BRSET_DATA_DIR  = r"C:\Users\preet\Documents\BRSET\data"
MBRSET_DATA_DIR = r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\data"
BRSET_IMG_ROOT  = r"C:\Users\preet\Documents\BRSET\data\resized_fundus_photos"
MBRSET_IMG_ROOT = r"C:\Users\preet\Documents\mBRSET\mbrset-a-mobile-brazilian-retinal-dataset-1.0\images"
OUTPUT_DIR      = "coverage_results"

## Load Data

In [ ]:
if DATASET == "mBRSET":
    val_df  = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_val_full.pkl"))
    test_df = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_test_full.pkl"))
    img_root   = MBRSET_IMG_ROOT
    num_images = 4

elif DATASET == "BRSET":
    val_df  = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_val_524.pkl"))
    test_df = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_test_524.pkl"))
    img_root   = BRSET_IMG_ROOT
    num_images = 2

patient_col = "patient"
for df in (val_df, test_df):
    df.rename(columns={"patient_id": "patient"}, inplace=True)
    df.dropna(subset=["final_icdr"], inplace=True)

print(f"Val rows: {len(val_df)}, Test rows: {len(test_df)}")

## Transforms

In [ ]:
if MODEL_NAME == "retfound_green":
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
else:
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

val_tf = A.Compose([
    A.Resize(392, 392),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

## Confidence Sweep Function

In [ ]:
def sweep_thresholds(all_probs, all_labels, T_step=0.01, prob_shift=0.0):
    """
    For each T in [0, 0.5), accept patients where p < 0.5-T or p > 0.5+T.
    Returns DataFrame with columns: T, coverage, BA, sensitivity, specificity.
    """
    probs    = np.clip(np.array(all_probs) - prob_shift, 0, 1)
    labels   = np.array(all_labels)
    T_values = np.arange(0.0, 0.5, T_step)
    rows = []

    for T in T_values:
        lo, hi = 0.5 - T, 0.5 + T
        mask = (probs < lo) | (probs > hi)
        n_accepted = mask.sum()
        coverage = n_accepted / len(probs)

        if n_accepted == 0:
            rows.append({"T": round(T, 4), "coverage": 0.0,
                         "BA": np.nan, "sensitivity": np.nan, "specificity": np.nan})
            continue

        p_acc, y_acc = probs[mask], labels[mask]
        pred_acc = (p_acc > 0.5).astype(int)

        tp = int(((pred_acc == 1) & (y_acc == 1)).sum())
        tn = int(((pred_acc == 0) & (y_acc == 0)).sum())
        fp = int(((pred_acc == 1) & (y_acc == 0)).sum())
        fn = int(((pred_acc == 0) & (y_acc == 1)).sum())

        sensitivity = tp / (tp + fn + 1e-8)
        specificity = tn / (tn + fp + 1e-8)

        rows.append({"T": round(T, 4), "coverage": round(coverage, 4),
                     "BA": round(0.5 * (sensitivity + specificity), 4),
                     "sensitivity": round(sensitivity, 4),
                     "specificity": round(specificity, 4)})

    return pd.DataFrame(rows)

## Find Checkpoints

In [ ]:
ckpt_pattern = f"{CHECKPOINT_PREFIX}_img_diagnosis_model_top*.pth"
checkpoints  = sorted(glob.glob(ckpt_pattern))[:N_CHECKPOINTS]
assert checkpoints, f"No checkpoints matched: {ckpt_pattern}"
print(f"Found {len(checkpoints)} checkpoint(s)")

device  = "cuda"
loss_fn = nn.CrossEntropyLoss()   # uniform weights — only probs/labels are used below

## Run Inference + Threshold Sweep

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

for ckpt_path in checkpoints:
    ckpt_name = os.path.splitext(os.path.basename(ckpt_path))[0]

    model = UnifiedBackboneMulti(model_name=MODEL_NAME, num_images=num_images)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device)

    for img_indices in IMG_INDICES_LIST:
        img_suffix = "imgs_" + ("all" if img_indices is None else "".join(str(i) for i in img_indices))

        val_ds  = MultiImageFundusDataset(val_df,  img_root, transform=val_tf, label_col="final_icdr",
                                          patient_col=patient_col, num_images=num_images, img_indices=img_indices)
        test_ds = MultiImageFundusDataset(test_df, img_root, transform=val_tf, label_col="final_icdr",
                                          patient_col=patient_col, num_images=num_images, img_indices=img_indices)
        val_loader  = DataLoader(val_ds,  batch_size=4, shuffle=False, num_workers=0)
        test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)

        for split, loader in [("val", val_loader), ("test", test_loader)]:
            _, metrics, *_ = validate_multi(model, loader, loss_fn, device)
            df_sweep = sweep_thresholds(metrics["all_probs"], metrics["all_labels"], T_STEP, prob_shift=PROB_SHIFT)
            out_csv = os.path.join(OUTPUT_DIR, f"{ckpt_name}_{img_suffix}_{split}_coverage.csv")
            df_sweep.to_csv(out_csv, index=False)

    print(f"Done: {ckpt_name}")

## Mean ± Std Summary (Test Set)

In [ ]:
split = "test"
img_suffixes = ["imgs_" + ("all" if ii is None else "".join(str(i) for i in ii)) for ii in IMG_INDICES_LIST]
plot_suffix  = "_".join(s.replace("imgs_", "") for s in img_suffixes)

csv_files = []
for ckpt_path in checkpoints:
    ckpt_name = os.path.splitext(os.path.basename(ckpt_path))[0]
    for img_suffix in img_suffixes:
        f = os.path.join(OUTPUT_DIR, f"{ckpt_name}_{img_suffix}_{split}_coverage.csv")
        if os.path.exists(f):
            csv_files.append(f)

assert csv_files, "No coverage CSVs found. Run the inference cell first."
print(f"Averaging over {len(csv_files)} CSV(s)")

dfs = [pd.read_csv(f).dropna().sort_values("coverage").reset_index(drop=True) for f in csv_files]
ref_coverage = dfs[0]["coverage"].values

def interp_col(ref_cov, df, col):
    return np.interp(ref_cov, df["coverage"].values, df[col].values)

ba_matrix   = np.stack([interp_col(ref_coverage, df, "BA")          for df in dfs])
sens_matrix = np.stack([interp_col(ref_coverage, df, "sensitivity") for df in dfs])
mean_ba,   std_ba   = ba_matrix.mean(axis=0),   ba_matrix.std(axis=0)
mean_sens, std_sens = sens_matrix.mean(axis=0), sens_matrix.std(axis=0)

bin_edges = np.linspace(ref_coverage.min(), ref_coverage.max(), N_BINS + 1)
bin_idx   = np.digitize(ref_coverage, bin_edges, right=True).clip(1, N_BINS) - 1

def bin_average(values):
    return np.array([values[bin_idx == b].mean() if (bin_idx == b).any() else np.nan for b in range(N_BINS)])

bin_coverage, bin_mean_ba, bin_std_ba = bin_average(ref_coverage), bin_average(mean_ba), bin_average(std_ba)
bin_mean_sens, bin_std_sens = bin_average(mean_sens), bin_average(std_sens)
valid = ~(np.isnan(bin_coverage) | np.isnan(bin_mean_ba) | np.isnan(bin_mean_sens))
bin_coverage, bin_mean_ba, bin_std_ba = bin_coverage[valid], bin_mean_ba[valid], bin_std_ba[valid]
bin_mean_sens, bin_std_sens = bin_mean_sens[valid], bin_std_sens[valid]

df_summary = pd.DataFrame({
    "coverage": ref_coverage, "BA_mean": mean_ba.round(4), "BA_std": std_ba.round(4),
    "sensitivity_mean": mean_sens.round(4), "sensitivity_std": std_sens.round(4),
})
summary_csv = os.path.join(OUTPUT_DIR, f"{CHECKPOINT_PREFIX}{plot_suffix}_test_mean_std_coverage.csv")
df_summary.to_csv(summary_csv, index=False)
print(f"Saved: {summary_csv}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle(f"{DATASET} ({plot_suffix}) — test set (mean ± std across {len(csv_files)} CSV(s))", fontsize=13)
for ax, mean_vals, std_vals, metric_name in zip(
    axes, [bin_mean_ba, bin_mean_sens], [bin_std_ba, bin_std_sens], ["Balanced Accuracy", "Sensitivity"]
):
    ax.plot(bin_coverage, mean_vals, color="steelblue", linewidth=2, label="mean")
    ax.errorbar(bin_coverage, mean_vals, yerr=std_vals, fmt="none", ecolor="steelblue", elinewidth=1.5, capsize=4, label="± 1 std")
    ax.set_xlabel("Coverage (fraction of patients accepted)")
    ax.set_ylabel(metric_name)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_title(metric_name)
plt.tight_layout()

mean_plot = os.path.join(OUTPUT_DIR, f"{CHECKPOINT_PREFIX}{plot_suffix}_test_mean_std_coverage_plot.png")
plt.savefig(mean_plot, dpi=150)
plt.show()
print(f"Saved: {mean_plot}")